In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
from pathlib import Path
import urllib.request


data_path = Path("names.txt")
if not data_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
    urllib.request.urlretrieve(url, data_path)

words = data_path.read_text().splitlines()
print(f"Loaded {len(words)} names from {data_path}")

Loaded 32033 names from names.txt


In [3]:
alphabets=sorted(list(set(''.join(words))))
stringToIndex={s:i+1 for i,s in enumerate(alphabets)}
IndexToString={s+1:i for s,i in enumerate(alphabets)}
stringToIndex["."]=0
IndexToString[0]="."
stringToIndex

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [4]:
# build the dataset
itos=IndexToString
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stringToIndex[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [5]:
print(X,Y)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]]) tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])


In [6]:
#lookup table 
C=torch.randn((27,2))
print(C)

tensor([[ 0.6158, -0.1068],
        [ 1.0089,  0.2027],
        [-1.3518, -0.9379],
        [ 0.8205,  0.8455],
        [-0.5003, -1.6107],
        [ 1.4968,  0.1555],
        [ 0.0215,  0.6088],
        [-1.4526, -0.9120],
        [-0.4875, -1.1850],
        [-0.0968,  0.8456],
        [-0.5899, -0.9782],
        [-0.5570, -1.2172],
        [ 0.1167,  0.5409],
        [ 1.9640,  1.3726],
        [-0.4962, -0.4381],
        [ 2.2046, -0.1995],
        [ 1.5196, -0.5092],
        [-1.0877,  1.8494],
        [ 0.0976, -1.9607],
        [-0.3623,  2.6222],
        [-0.4372, -1.4745],
        [-0.9037,  0.3127],
        [-1.4638, -0.3180],
        [ 1.3162,  0.7205],
        [ 0.3918, -0.0699],
        [ 0.3927, -0.7379],
        [ 0.6981,  0.3093]])


In [7]:
(F.one_hot(torch.tensor(5),num_classes=27).float())@C

tensor([1.4968, 0.1555])

In [8]:
C[5]

tensor([1.4968, 0.1555])

In [9]:
embeddings=C[X]

In [10]:
print(embeddings)

tensor([[[ 0.6158, -0.1068],
         [ 0.6158, -0.1068],
         [ 0.6158, -0.1068]],

        [[ 0.6158, -0.1068],
         [ 0.6158, -0.1068],
         [ 1.4968,  0.1555]],

        [[ 0.6158, -0.1068],
         [ 1.4968,  0.1555],
         [ 1.9640,  1.3726]],

        [[ 1.4968,  0.1555],
         [ 1.9640,  1.3726],
         [ 1.9640,  1.3726]],

        [[ 1.9640,  1.3726],
         [ 1.9640,  1.3726],
         [ 1.0089,  0.2027]],

        [[ 0.6158, -0.1068],
         [ 0.6158, -0.1068],
         [ 0.6158, -0.1068]],

        [[ 0.6158, -0.1068],
         [ 0.6158, -0.1068],
         [ 2.2046, -0.1995]],

        [[ 0.6158, -0.1068],
         [ 2.2046, -0.1995],
         [ 0.1167,  0.5409]],

        [[ 2.2046, -0.1995],
         [ 0.1167,  0.5409],
         [-0.0968,  0.8456]],

        [[ 0.1167,  0.5409],
         [-0.0968,  0.8456],
         [-1.4638, -0.3180]],

        [[-0.0968,  0.8456],
         [-1.4638, -0.3180],
         [-0.0968,  0.8456]],

        [[-1.4638, -0

In [11]:
W1=torch.randn(6,100)
b1=torch.randn(100)

In [14]:
from torch import tanh


h=tanh(embeddings.view(-1,6)@W1+b1)
h

tensor([[ 0.9312,  0.8736,  0.9288,  ...,  0.9729,  0.5183,  0.0975],
        [ 0.9888,  0.9593,  0.9976,  ...,  0.9861,  0.7106,  0.6834],
        [ 0.9688,  0.9253,  1.0000,  ...,  0.9960,  0.9912,  0.9930],
        ...,
        [ 0.9984,  0.3667,  0.6024,  ...,  0.9983,  0.3910, -0.9470],
        [ 1.0000, -0.9911,  0.2480,  ...,  0.9992, -0.6335, -0.6582],
        [-0.9973,  0.9868,  0.7875,  ...,  0.9544, -0.4126,  0.9836]])

In [15]:
W2=torch.randn(100,27)
b2=torch.randn(27)

In [18]:
logits=h@W2+b2
logits

tensor([[-1.8860e+00,  7.9595e+00,  3.9825e+00,  6.7269e+00,  1.0566e+01,
          8.1801e+00,  4.6826e+00, -8.5576e+00,  1.0243e+01,  7.1275e+00,
         -5.2982e+00,  6.2365e+00, -5.2176e+00,  5.2565e+00,  3.2956e+00,
         -8.4479e+00,  3.3731e+00, -9.2992e+00, -1.2603e+01,  6.5987e+00,
         -6.8369e+00,  1.1692e+01,  2.1047e+00, -4.7150e+00,  1.8925e+00,
          3.5423e+00, -4.6262e+00],
        [-4.6221e+00,  4.8199e+00,  2.1296e+00,  9.1451e+00,  8.7157e+00,
          8.5187e+00,  1.9256e+00, -1.3847e+01,  6.1692e+00,  1.0493e+01,
         -6.5729e+00,  3.4465e+00, -4.8103e+00,  7.4184e+00,  6.0204e+00,
         -7.2793e+00,  7.4176e+00, -7.8436e+00, -7.6625e+00,  3.3593e+00,
         -6.4032e+00,  9.7191e+00, -1.7643e+00, -8.2339e+00,  1.0131e+01,
          6.2257e+00, -2.1440e+00],
        [-1.3657e+01, -2.3166e+00, -6.7235e-01, -2.9397e+00,  4.1982e+00,
         -5.1522e-01, -1.6993e+00, -5.3200e+00,  1.9603e+00,  7.6960e+00,
         -5.3849e+00, -2.3875e+00, -1.34

In [19]:
counts=logits.exp()

In [21]:
probs=counts/counts.sum(1,keepdim=True)
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])